In [1]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import pandas as pd
from pathlib import Path
import os
import json
import datetime as dt
from ftplib import FTP_TLS
import tkinter as tk
from tkinter import messagebox
import csv
from openpyxl import load_workbook

from pydataxm import *                           #Se realiza la importación de las librerias necesarias para ejecutar                        
from pydataxm.pydataxm import ReadDB as apiXM 
import datetime as dt  

from google.cloud import bigquery
from google.oauth2 import service_account

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)  # ✅ Hace que los messagebox estén en primer plano

''

### Descargar archivos programado

In [5]:
def readfileOfe(file_path):
    # Read all lines from the file
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
        
    # Initialize variables
    data = []
    current_agent = None

    # Process each line
    for line in lines:
        line = line.strip()

        if not line:
            continue  # Skip empty lines

        if line.startswith("AGENTE:"):
            current_agent = line.replace("AGENTE:", "").strip()
            continue

        parts = [x.strip() for x in line.split(",")]

        if len(parts) >= 3:
            unidad = parts[0]
            tipo = parts[1]
            valores = parts[2:]

            # Try to convert values to float; skip line if fails
            if len(valores)>6:
                valores_float = [float(v) for v in valores]
            else:
                if valores[0].replace('.', '', 1).isdigit():
                    valores_float = [float(v) for v in valores]
                else:
                    valores_float = [v for v in valores]
                

            for hora, valor in enumerate(valores_float, start=1):
                data.append({
                    "Agente": current_agent,
                    "Unidad": unidad,
                    "Tipo": tipo,
                    "Hora": hora,
                    "Valor": valor
                })

    # Convert to DataFrame
    df = pd.DataFrame(data)

    return df

# Show the first few rows (optional)
# print(df.head())

# df_filter= df[(df['Tipo'].str.match(r'^P\d+')) & (df['Unidad']=='TEBSABCC')]

In [ ]:
# Buscar el precio de cada una de las plantas
def PreciosPlt(df):
    # Lectura de datos con los mapeos
    s_parentpath=Path('C:\Alejo\cops\Data')
    filepath=s_parentpath.joinpath(s_parentpath,'Mapeos.xlsx')


    l_sheets=['NomRecursos','RecHidraulicos','RecVariable']

    df_precios_plt=pd.DataFrame()
    df_plantas=pd.DataFrame()

    for sheet_name in l_sheets:
        
        df_pltt=pd.read_excel(filepath, header=0,sheet_name=sheet_name)
        df_plantas=pd.concat([df_plantas,df_pltt],axis=0)
        
        if sheet_name=='NomRecursos':
            # Plantas térmica
            df_plt=df_pltt[(df_pltt.TipoPlt=='CS')][['RecCops','RecOfe','TipoPlt']]
            df_plt['Poferta']=0

            df_plt.loc[df_plt['RecOfe'].str.contains('PROELECTRICA1|PROELECTRICA2', regex=True), 'RecOfe'] = 'PROELECTRICA'

        elif sheet_name=='RecHidraulicos' or sheet_name=='RecVariable':
            df_plt=df_pltt[['RecCops','RecOfe','TipoPlt']]
            df_plt['Poferta']=0
            df_plt['Confoferta']=1

        for ind in df_plt.index:
            plt=df_plt.loc[ind,'RecOfe']
            # print(plt)
            if plt in df['Unidad'].values:
                df_filter= df[(df['Unidad']==plt) & (df['Tipo']=='P')]
                if df_filter.shape[0]>0:  
                    df_filter= df[(df['Unidad']==plt) & (df['Tipo']=='P')]['Valor'].values[0]
                    df_plt.loc[ind,'Poferta']=df_filter
                else:
                    messagebox.showinfo('Información proceso de oferta',rf'El recurso {plt} no tienen precio de oferta', parent=root)
                    df_plt.loc[ind,'Poferta']=0

                if df_plt.loc[ind,'TipoPlt']=='CS':
                    df_filter= df[(df['Unidad']==plt) & (df['Tipo']=='CONF')]
                    if df_filter.shape[0]>0:  
                        df_filter= df[(df['Unidad']==plt) & (df['Tipo']=='CONF')]['Valor'].values[0]
                        df_plt.loc[ind,'Confoferta']=df_filter
                    else:
                        messagebox.showinfo('Información proceso de oferta',rf'El recurso {plt} no tienen configuración de oferta', parent=root)
                        df_plt.loc[ind,'Confoferta']=1
                else:
                    df_plt.loc[ind,'Confoferta']=0

                
            else:
                print('No se encontró la planta:', plt)

        df_precios_plt=pd.concat([df_precios_plt,df_plt],axis=0)
        
    df_precios_plt=df_precios_plt[['RecCops','Poferta','Confoferta']]
    df_precios_plt['Poferta']=df_precios_plt['Poferta']/1000

    df_precios_plt = pd.concat([df_precios_plt, pd.DataFrame({'RecCops': ['FLORES_I_CC'], 'Poferta': [10002], 'Confoferta': [1]})], ignore_index=True)
    df_precios_plt = pd.concat([df_precios_plt, pd.DataFrame({'RecCops': ['FLORES_4_CC'], 'Poferta': [10003], 'Confoferta': [1]})], ignore_index=True)
    df_precios_plt = pd.concat([df_precios_plt, pd.DataFrame({'RecCops': ['TEBSAB_CC'], 'Poferta': [10000], 'Confoferta': [1]})], ignore_index=True)
    df_precios_plt = pd.concat([df_precios_plt, pd.DataFrame({'RecCops': ['TERMOCANDELARIA_CC'], 'Poferta': [10001], 'Confoferta': [1]})], ignore_index=True)
    df_precios_plt = pd.concat([df_precios_plt, pd.DataFrame({'RecCops': ['TERMOCENTRO_CC'], 'Poferta': [10004], 'Confoferta': [1]})], ignore_index=True)
    df_precios_plt = pd.concat([df_precios_plt, pd.DataFrame({'RecCops': ['TERMOEMCALI_CC'], 'Poferta': [10005], 'Confoferta': [1]})], ignore_index=True)
    df_precios_plt = pd.concat([df_precios_plt, pd.DataFrame({'RecCops': ['TERMOSIERRA_CC'], 'Poferta': [10006], 'Confoferta': [1]})], ignore_index=True)
    df_precios_plt = pd.concat([df_precios_plt, pd.DataFrame({'RecCops': ['TERMOVALLE_CC'], 'Poferta': [10007], 'Confoferta': [1]})], ignore_index=True)
    
    
    return df_precios_plt,df_plantas


In [7]:
def PreciosPltCC(df):
    df_precios_plt=pd.DataFrame()
    # Carga del nivel probabilístico del embalse
    s_parentpath=Path('C:\Alejo\cops\Data')
    filepath=s_parentpath.joinpath(s_parentpath,'Mapeos.xlsx')
    sheet_name='NomRecursos'
    df_pltt=pd.read_excel(filepath, header=0,sheet_name=sheet_name)
    df_plt=df_pltt[(df_pltt.TipoPlt=='CC')][['RecCops','RecOfe']]
    df_plt_final=df_plt.copy()

    for ind in df_plt.index:
        plt=df_plt.loc[ind,'RecOfe']
        if plt in df['Unidad'].values:
            df_filter = df[(df['Unidad'] == plt) & (df['Tipo'].str.startswith('P'))][['Unidad','Tipo','Valor']]
            df_plt_final = df_filter.merge(df_plt, left_on=['Unidad'],right_on=['RecOfe'], how='left')[['RecCops','RecOfe','Tipo','Valor']]
            df_plt_final['Tipo_num'] = df_plt_final['Tipo'].str.extract(r'(\d+)').astype(float)
            df_plt_final = df_plt_final.sort_values(by='Tipo_num')
            df_plt_final['Conf'] = df_plt_final['Tipo_num'].astype(int)
            df_plt_final['Conf']=".c" + df_plt_final['Conf'].astype(str)
            df_plt_final = df_plt_final.drop(columns=['Tipo_num'])
            # df_plt_final = df_plt_final.sort_values(by='Tipo')
            # df_filter= df[(df['Unidad']==plt) & (df['Tipo']=='P')]['Valor']
        else:
            print('No se encontró la planta:', plt)

        df_precios_plt=pd.concat([df_precios_plt,df_plt_final],axis=0)

    df_precios_plt=df_precios_plt[['RecCops','Conf','Valor']]
    df_precios_plt['Valor']=df_precios_plt['Valor']/1000
    
    return df_precios_plt

In [8]:
def DispUni(df):
    # Lectura de datos con los mapeos
    s_parentpath=Path('C:\Alejo\cops\Data')
    filepath=s_parentpath.joinpath(s_parentpath,'Mapeos.xlsx')

    df=df[(df['Tipo']=='D')]

    l_sheets=['NomUnidad','UniHidro','UniVariable']

    df_precios_plt=pd.DataFrame()

    # --- Preprocesar df como diccionario para búsqueda rápida
    df_lookup = df.set_index(['Unidad', 'Hora'])['Valor'].to_dict()
    unidades_validas = set(df['Unidad'].unique())

    # --- Procesamiento por hoja
    for sheet_name in l_sheets:
        df_pltt = pd.read_excel(filepath, sheet_name=sheet_name)
        df_plt = df_pltt[['UniCops', 'UniOfe','Tipo']].copy()

        # Inicializar columnas D_1 a D_24
        l_per=[]
        for i in range(1, 25):
            df_plt[f'{i}'] = 0
            l_per.append(f'{i}')

        # Llenar valores por unidad y hora
        for ind, row in df_plt.iterrows():
            unidad = row['UniOfe']

            if unidad in unidades_validas:
                for hora in range(1, 25):
                    valor = df_lookup.get((unidad, hora), None)
                    if valor is not None:
                        df_plt.at[ind, f'{hora}'] = valor
            else:
                print(f'No se encontró la planta: {unidad}')

        # Concatenar resultados
        df_precios_plt = pd.concat([df_precios_plt, df_plt], axis=0, ignore_index=True)

    df_precios_plt=df_precios_plt[['UniCops'] + l_per ]

    return df_precios_plt

In [9]:
def DispCC(df):
    # Lectura de datos con los mapeos
    s_parentpath=Path('C:\Alejo\cops\Data')
    filepath=s_parentpath.joinpath(s_parentpath,'Mapeos.xlsx')

    df_pltt = pd.read_excel(filepath, sheet_name='NomRecursos')
    df_pltcc = df_pltt[['RecCops','RecOfe','RecDesp','TipoPlt']]
    df_pltcc=df_pltcc[(df_pltcc.TipoPlt=='CC')]
    unique_plt=list(df_pltcc['RecOfe'].unique())

    df_cc = df[(df['Unidad'].isin(unique_plt)) & (df['Tipo'].str.startswith('DISCONF'))][['Unidad','Tipo','Hora','Valor']]

    df_maxConf=df_cc[(df_cc.Hora==1)]
    # df_maxConf= df_maxConf.loc[df_maxConf.groupby("Unidad")["Valor"].idxmax(), ["Unidad", "Tipo", "Valor"]]

    df_precios_plt=pd.DataFrame()

    # --- Preprocesar df como diccionario para búsqueda rápida
    df_lookup = df_cc.set_index(['Unidad','Tipo','Hora'])['Valor'].to_dict()

    df_plt=df_maxConf.copy()
    df_plt=df_plt[['Unidad','Tipo']]

    # Inicializar columnas D_1 a D_24
    l_per = []
    for i in range(1, 25):
        df_plt[f'{i}'] = 0
        l_per.append(f'{i}')

    # Llenar valores por unidad y hora
    for ind, row in df_maxConf.iterrows():
        unidad = row['Unidad']
        conf = row['Tipo']

        for hora in range(1, 25):
            valor = df_lookup.get((unidad,conf, hora), None)
            if valor is not None:
                df_plt.at[ind, f'{hora}'] = valor


        # Concatenar resultados
        # df_precios_plt = pd.concat([df_precios_plt, df_plt], axis=0, ignore_index=True)

    l_col=list(df_plt.columns)
    l_col.append('RecOfe')
    l_col.append('RecCops')
    df_plt=df_plt.merge(df_pltcc,left_on=['Unidad'],right_on=['RecOfe'],how='inner')[l_col]

    df_plt['Tipo_num'] = df_plt['Tipo'].str.extract(r'(\d+)').astype(float)
    df_plt = df_plt.sort_values(by='Tipo_num')
    df_plt['Conf'] = df_plt['Tipo_num'].astype(int)
    df_plt['Conf']=".c" + df_plt['Conf'].astype(str)
    df_plt = df_plt.drop(columns=['Tipo_num'])
    
    df_plt=df_plt[['RecCops','Conf'] + l_per ]
    
    return df_plt

In [10]:
def MOPlt(df):
    # Lectura de datos con los mapeos
    s_parentpath=Path('C:\Alejo\cops\Data')
    filepath=s_parentpath.joinpath(s_parentpath,'Mapeos.xlsx')

    df=df[(df['Tipo']=='MO')]

    l_sheets=['RecHidraulicos']

    df_precios_plt=pd.DataFrame()

    # --- Preprocesar df como diccionario para búsqueda rápida
    df_lookup = df.set_index(['Unidad', 'Hora'])['Valor'].to_dict()
    unidades_validas = set(df['Unidad'].unique())

    # --- Procesamiento por hoja
    for sheet_name in l_sheets:
        df_pltt = pd.read_excel(filepath, sheet_name=sheet_name)
        df_plt = df_pltt[['RecCops', 'RecOfe','TipoPlt']].copy()

        df_plt=df_plt[(df_plt['RecOfe'].isin(unidades_validas))]

        # Inicializar columnas D_1 a D_24
        l_per = []
        for i in range(1, 25):
            df_plt[f'{i}'] = 0
            l_per.append(f'{i}')

        # Llenar valores por unidad y hora
        for ind, row in df_plt.iterrows():
            unidad = row['RecOfe']

            if unidad in unidades_validas:
                for hora in range(1, 25):
                    valor = df_lookup.get((unidad, hora), None)
                    if valor is not None:
                        df_plt.at[ind, f'{hora}'] = valor
            else:
                print(f'No se encontró la planta: {unidad}')

        # Concatenar resultados
        df_precios_plt = pd.concat([df_precios_plt, df_plt], axis=0, ignore_index=True)

    df_precios_plt=df_precios_plt[['RecCops'] + l_per ]
        
    return df_precios_plt

In [11]:
def PruebasUni(df):
    # Lectura de datos con los mapeos
    s_parentpath=Path('C:\Alejo\cops\Data')
    filepath=s_parentpath.joinpath(s_parentpath,'Mapeos.xlsx')

    df=df[(df['Tipo']=='PRU')]

    df_p=df.groupby(['Unidad'])[['Valor']].sum().reset_index()
    df_p = df_p[df_p['Valor'].astype(float) > 0]

    l_sheets=['NomUnidad','UniHidro','UniVariable']

    df_precios_plt=pd.DataFrame()

    # --- Preprocesar df como diccionario para búsqueda rápida
    df_lookup = df.set_index(['Unidad', 'Hora'])['Valor'].to_dict()
    unidades_validas = set(df_p['Unidad'].unique())

    # --- Procesamiento por hoja
    for sheet_name in l_sheets:
        df_pltt = pd.read_excel(filepath, sheet_name=sheet_name)
        df_plt = df_pltt[['UniCops', 'UniOfe','Tipo']].copy()
        df_plt=df_plt[(df_plt['UniOfe'].isin(unidades_validas))]

        # Inicializar columnas D_1 a D_24
        l_per = []
        for i in range(1, 25):
            df_plt[f'{i}'] = 0
            l_per.append(f'{i}')

        # Llenar valores por unidad y hora
        for ind, row in df_plt.iterrows():
            unidad = row['UniOfe']

            if unidad in unidades_validas:
                for hora in range(1, 25):
                    valor = df_lookup.get((unidad, hora), None)
                    if valor is not None:
                        df_plt.at[ind, f'{hora}'] = valor
            else:
                print(f'No se encontró la planta: {unidad}')

        # Concatenar resultados
        df_precios_plt = pd.concat([df_precios_plt, df_plt], axis=0, ignore_index=True)

    df_precios_plt = df_precios_plt[['UniCops'] + l_per]

    return df_precios_plt

In [12]:

# Función para descargar el archivo
def DownFile(fecha_dt,UsuXM,PwsXM,Tipo):

    # Connect to the FTP server (replace with your actual details)
    ftps  = FTP_TLS()
    ftps .connect('xmftps.xm.com.co', 210)  # default port is 210

    # Secure the control connection
    ftps .auth()
    ftps .prot_p()  # Switch to secure data connection (important!)

    ftps .login(UsuXM, PwsXM)

    # Obtener mes y día de la fecha inicial
    # Transformar string en fecha
    # fecha = dt.datetime.strptime(fecha_dt, "%d/%m/%Y")

    # Obtener mes y día
    year= fecha_dt.year
    mes = fecha_dt.month
    dia = fecha_dt.day

    if Tipo=='Oferta':
        # Navigate to the directory you want to access
        ftps.cwd(rf"/INFORMACION_XM/PUBLICO/OFERTAS/INICIAL/{year:04d}-{mes:02d}")
    else:
        messagebox.showinfo('Estado del proceso',f'No se reconoce el formato {Tipo}', parent=root)
        df=pd.DataFrame()
        return df


    # List files
    files = ftps.nlst()
    # print("Available files:", files)


    if Tipo=='Oferta':
        # Download condiciones iniciales de planta
        pathfile=rf"C:\Información XM\PUBLICO\OFERTAS\INICIAL\{year:04d}-{mes:02d}\\"
        filename = f"OFEI{mes:02d}{dia:02d}.TXT"      
    
    print(pathfile + filename)
    with open(pathfile + filename, 'wb') as f:
        ftps.retrbinary(f"RETR {filename}", f.write)

    ftps.quit()
    # print(f"{filename} downloaded successfully.")

    if Tipo=='Oferta':
        df=readfileOfe(pathfile + filename)
    
    return df


In [4]:
# try:
# Ruta del archivo
sFile=r"Parametros.json"
script_dir = Path.cwd()
script_dir=script_dir.parent.parent
sPathfile=os.path.join(script_dir,r"Modules\Utils\ArchivosAux",sFile)
sPathfile = os.path.join(r"C:\Alejo\cops\Modules\Utils\ArchivosAux",sFile)

# Open and load the JSON file
with open(sPathfile,'r') as f:
    data = json.load(f)

# Almancenar los parámetros en variables python
year=data['Parametros']['Ano']
mes=data['Parametros']['Mes']
Carpeta=data['Parametros']['Carpeta']
FechaInicial=data['Parametros']['Fecha_Inicial']
FechaFinal=data['Parametros']['Fecha_Final']
FileName=data['Parametros']['Nombre_Archivo']
FechaDem=data['Parametros']['Fecha_Demanda']

# Banderas para la ejecución
DemFile=data['BanderasFile']['CB_ArchivoDem']
AccessFile=data['BanderasFile']['CB_ArchivoAccess']
InfoTipoDia=data['BanderasFile']['CB_TipoDia']

# Banderas carga de precios
b_PrecioPlt=data['BanderasLoadPrecios']['CB_LoadPrecios']
b_PrecioTPL=data['BanderasLoadPrecios']['CB_LoadPreciosTPL']

if b_PrecioPlt=='Verdadero':
    b_PrecioPlt=1
else:
    b_PrecioPlt=0

if b_PrecioTPL=='Verdadero':
    b_PrecioTPL=1
else:
    b_PrecioTPL=0



UsuXM=data['Pws']['UsuarioXm']
PwsXM=data['Pws']['PwsXm']

# Leer Mapeo planta unidad
spathFileMap=r'C:\Alejo\cops\Data\Mapeos.xlsx'
mapping_df=pd.read_excel(spathFileMap,sheet_name='Planta_Unidad',header=0)


# Descargar archivo de oferta para el día 1 y obtener los precios iniciales

# Obtener la fecha del día 1
fecha_dt = dt.datetime.strptime(FechaInicial, "%d/%m/%Y")
diaini = fecha_dt.day
mesini=fecha_dt.month
yearini=fecha_dt.year



if 1==0:
    # Descargar archivo de la oferta
    df=DownFile(fecha_dt,UsuXM,PwsXM,Tipo='Oferta')
    # Precios de las plantas
    df_precios,df_plantas=PreciosPlt(df)
    df_plantascc=list(df_plantas[(df_plantas.TipoPlt=='CC')]['RecOfe'])

    # Precios de las plantas de CC
    df_preciosCC=PreciosPltCC(df)
    # Disponibilidad Unidad
    df_DispUni=DispUni(df)
    # Disponibilidad por configuración
    df_DispCC=DispCC(df)

    # Recursos con mínimo obligatorio
    df_MO=MOPlt(df)

    # Pruebas de unidad
    df_Pru=PruebasUni(df)



    df_precios.to_csv(os.path.join(r"C:\Alejo\cops\Modules\Utils\InformacionDia", f"Precios_plantas_{yearini}_{mesini}_{diaini}.csv"), index=False)
    df_preciosCC.to_csv(os.path.join(r"C:\Alejo\cops\Modules\Utils\InformacionDia", f"Precios_plantasCC_{yearini}_{mesini}_{diaini}.csv"), index=False)
    df_DispUni.to_csv(os.path.join(r"C:\Alejo\cops\Modules\Utils\InformacionDia", f"Disponibilidad_Unidad_{yearini}_{mesini}_{diaini}.csv"), index=False)
    df_DispCC.to_csv(os.path.join(r"C:\Alejo\cops\Modules\Utils\InformacionDia", f"Disponibilidad_CC_{yearini}_{mesini}_{diaini}.csv"), index=False)
    df_MO.to_csv(os.path.join(r"C:\Alejo\cops\Modules\Utils\InformacionDia", f"MO_plantas_{yearini}_{mesini}_{diaini}.csv"), index=False)
    df_Pru.to_csv(os.path.join(r"C:\Alejo\cops\Modules\Utils\InformacionDia", f"Pruebas_unidades_{yearini}_{mesini}_{diaini}.csv"), index=False)


    # messagebox.showinfo('Estado del proceso','Se imprimió el archivo de excel de manera correcta', parent=root)

# except:

#     messagebox.showerror('Estado del proceso','Error en el proceso, por favor validar', parent=root)  

### Descargar archivos real

In [2]:
# Información del proyecto y autenticación a BQ
project_id = "enersinc-tbsg-bq"
key_path = "C:\BigQuery\eramirez-tbsg.json"

# Cargar las credenciales del archivo JSON
credentials = service_account.Credentials.from_service_account_file(key_path)

# Crear el cliente de BigQuery
client = bigquery.Client(project=project_id, credentials=credentials)


In [ ]:
FechaIni=pd.to_datetime(FechaInicial)
version='txf'
# Consulta a la maestra de recursos
query = rf"""
select * 
from tbsg.public_grip 
where fechaoperacion='{FechaIni.strftime('%Y-%m-%d')}' and tipo='DCOM' and version = '{version}'
"""

# Ejecutar la consulta
df_DCOMIni = client.query(query).to_dataframe()


# Consulta a la maestra de recursos
query = """
select distinct *
from tbsg.public_maestra_recurso 
where recurso_despacho is not null
"""
# Ejecutar la consulta
df_MaestraRec= client.query(query).to_dataframe()

In [6]:
FechaIni=dt.date(FechaIni.year, FechaIni.month, FechaIni.day)
FechaFin=dt.date(FechaIni.year, FechaIni.month, FechaIni.day)

In [64]:
df_RecIni= apiXM.request_data(pydataxm.ReadDB(),    #Se indica el objeto que contiene el serivicio
                        "ListadoRecursos",                #Se indica el nombre de la métrica tal como se llama en el campo metricID
                        "Sistema",                 #Campo que indica si es Sistema, Agente, Recurso, Comercializador
                        FechaIni,       #Corresponde a la fecha inicial de la consulta
                        FechaFin)      #Corresponde a la fecha final de la consulta
df_RecIni=df_RecIni.drop('Date',axis=1)

df_RecIni.to_csv('ListadoRecursos.csv')

In [8]:
# Lectura de datos con los mapeos
s_parentpath=Path('C:\Alejo\cops\Data')
filepath=s_parentpath.joinpath(s_parentpath,'Mapeos.xlsx')

df_plt_sub=pd.read_excel(filepath,sheet_name='Menores')
df_plt_sub=pd.concat([df_plt_sub,pd.read_excel(filepath,sheet_name='DC')],axis=0)


l_sheets=['NomRecursos','RecHidraulicos','RecVariable']
df_plantas=pd.DataFrame()

for sheet_name in l_sheets:
    
    df_pltt=pd.read_excel(filepath, header=0,sheet_name=sheet_name)
    df_plantas=pd.concat([df_plantas,df_pltt],axis=0)

In [36]:
df_DCOM=df_DCOMIni.copy()
df_Rec=df_RecIni.copy()

l_per=[]
for i in range(1,25):
    df_DCOM = df_DCOM.rename(columns={f'hora{i}': i})
    df_DCOM[i]=(df_DCOM[i]/1000).round(4)
    l_per.append(i)
    # if i<10:
    #     df_ConfD = df_ConfD.rename(columns={f'p0{i}': i})
    # else:
    #     df_ConfD = df_ConfD.rename(columns={f'p{i}': i}) 

# df_DCOM = df_DCOM.melt(id_vars=['fechaoperacion', 'planta'], 
#                        value_vars=l_per, 
#                        var_name='periodo', 
#                        value_name='dcom')

df_DCOM=df_DCOM.merge(df_Rec,left_on=['planta'],right_on=['Values_Code'], 
                        how='inner')[['fechaoperacion','planta','Values_Name','Values_Type','Values_Disp'] + l_per]

# df_DCOM_null_values = df_DCOM[df_DCOM['Values_Name'].isnull()]

l_col=list(df_DCOM.columns)

l_col.append('Subarea')

df_DCOM = df_DCOM.merge(df_plt_sub[['Planta','Subarea']], left_on='Values_Name', right_on='Planta', how='left')
l_col.append('subarea')
df_MaestraRec = df_MaestraRec.drop_duplicates(subset='codsic_planta')
df_DCOM=df_DCOM.merge(df_MaestraRec[['codsic_planta','subarea']],left_on=['planta'],right_on=['codsic_planta'], how='left')[l_col]
df_DCOM['Subarea'] = df_DCOM['Subarea'].fillna(df_DCOM['subarea'])
df_DCOM['Subarea'] = df_DCOM['Subarea'].fillna('No tiene')
df_DCOM['Subarea'] = df_DCOM['Subarea'].replace({
       'SubArea Cordoba_Sucre':'CORDOSUC',
       'SubArea Atlantico':'ATLANTIC',
       'SubArea Cerromatoso':'CERROMAT',
       'SubArea GCM':'GCM',
       'SubArea Bolivar':'BOLIVAR'
})
df_DCOM['subareafinal'] = df_DCOM['Subarea'].apply(lambda x: x if x in ['ATLANTIC','BOLIVAR','GCM','CERROMAT','CORDOSUC'] else 'INTERIOR')
df_DCOM.drop(columns=['Subarea','subarea'], inplace=True)

l_col=list(df_DCOM.columns)
l_col.append('RecCops')
df_DCOM=df_DCOM.merge(df_plantas[['RecCops','Sinergox']], left_on='Values_Name', right_on='Sinergox', how='left')[l_col]
df_DCOM

,fechaoperacion,planta,Values_Name,Values_Type,Values_Disp,1,2,3,4,5,...,17,18,19,20,21,22,23,24,subareafinal,RecCops
0,2025-08-28,4QPJ,SOL DEL MAR II,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,6.2410,1.6176,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,CORDOSUC,NaN
1,2025-08-28,4RG6,AUTOG CELSIA SOLAR TQ JAMUNDI,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,INTERIOR,NaN
2,2025-08-28,4RGG,PARQUE SOLAR BARANOA,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,6.4200,2.2950,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,ATLANTIC,NaN
3,2025-08-28,4RMR,MINIGRANJA LA PAZ ESMERALDA,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.1523,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,GCM,NaN
4,2025-08-28,4S1G,MINIGRANJA VILLANUEVA,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.4937,0.0228,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,GCM,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417,2025-08-28,2QEK,SALTO II,HIDRAULICA,DESPACHADO CENTRALMENTE,35.0000,35.0000,35.0000,35.0000,35.0000,...,35.0000,35.0000,35.0000,35.0000,35.0000,35.0000,35.0000,35.0000,INTERIOR,SALTO_II
418,2025-08-28,2S8N,GUAVIO MENOR,HIDRAULICA,NO DESPACHADO CENTRALMENTE,5.0081,5.0249,5.0428,5.3112,5.3233,...,5.0748,4.9933,5.0089,5.0102,5.0156,5.0212,5.0207,4.9774,INTERIOR,NaN
419,2025-08-28,2S6S,AUTOG ARGOS YUMBO,TERMICA,NO DESPACHADO CENTRALMENTE,7.0380,8.2380,6.7650,5.5260,5.7090,...,4.7550,4.5150,4.4850,4.3050,4.4430,5.2650,5.5740,5.5530,INTERIOR,NaN
420,2025-08-28,2QRL,LA REBUSCA,HIDRAULICA,NO DESPACHADO CENTRALMENTE,0.6632,0.6635,0.6631,0.5850,0.6602,...,0.6592,0.6591,0.6590,0.6588,0.6588,0.6588,0.6590,0.6590,INTERIOR,NaN


In [59]:
# Plantas térmica con despacho central
df_final=pd.DataFrame()
df_Aux=df_DCOM[(df_DCOM.Values_Type=='TERMICA') & (df_DCOM.Values_Disp=='DESPACHADO CENTRALMENTE')][['RecCops'] + l_per]
df_final=pd.concat([df_final,df_Aux],axis=0)
df_Aux=df_DCOM[(df_DCOM.Values_Type=='HIDRAULICA') & (df_DCOM.Values_Disp=='DESPACHADO CENTRALMENTE')][['RecCops'] + l_per]
df_final=pd.concat([df_final,df_Aux],axis=0)

df_Aux

,RecCops,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
60,ALBAN,376.0,376.0,376.0,376.0,376.0,376.0,376.0,376.0,376.00,...,400.0,400.0,400.00,400.00,400.00,400.00,400.0,400.0,400.0,400.0
76,BETANIA,495.0,495.0,495.0,495.0,495.0,495.0,495.0,495.0,495.00,...,495.0,495.0,495.00,495.00,495.00,495.00,495.0,495.0,495.0,495.0
77,CHIVOR,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.0,500.00,...,500.0,500.0,500.00,500.00,500.00,500.00,500.0,500.0,500.0,500.0
81,CARLOS_LLERAS,54.0,54.0,54.0,54.0,54.0,54.0,54.0,54.0,46.79,...,60.0,60.0,60.00,60.00,42.00,42.00,42.0,45.0,55.0,55.0
82,CALIMA,132.0,132.0,132.0,132.0,132.0,132.0,132.0,132.0,132.00,...,132.0,132.0,132.00,132.00,132.00,132.00,132.0,132.0,132.0,132.0
93,CUCUANA,30.0,30.0,30.0,30.0,30.0,30.0,30.0,30.0,30.00,...,50.0,50.0,50.00,50.00,50.00,50.00,50.0,50.0,50.0,50.0
95,DARIO_VALENCIA_SAMPER,150.0,150.0,150.0,150.0,150.0,100.0,100.0,100.0,100.00,...,100.0,100.0,100.00,100.00,100.00,100.00,100.0,150.0,150.0,150.0
97,ESMERALDA,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.0,15.00,...,15.0,15.0,15.00,15.00,15.00,15.00,15.0,15.0,15.0,15.0
104,GUATAPE,490.0,490.0,490.0,490.0,490.0,490.0,490.0,490.0,490.00,...,490.0,490.0,490.00,490.00,490.00,490.00,490.0,490.0,490.0,490.0
105,GUATRON,409.0,409.0,409.0,409.0,409.0,409.0,409.0,409.0,409.00,...,409.0,409.0,398.62,394.87,393.58,393.43,409.0,392.0,440.0,440.0


In [60]:
# Apply the filter to discard rows where both conditions do not match simultaneously
df_Variable = df_DCOM[~((df_DCOM['Values_Type'] == 'HIDRAULICA') & (df_DCOM['Values_Disp'] == 'DESPACHADO CENTRALMENTE')) & 
                             ~((df_DCOM['Values_Type'] == 'TERMICA') & (df_DCOM['Values_Disp'] == 'DESPACHADO CENTRALMENTE'))]
df_Variable

,fechaoperacion,planta,Values_Name,Values_Type,Values_Disp,1,2,3,4,5,...,17,18,19,20,21,22,23,24,subareafinal,RecCops
0,2025-08-28,4QPJ,SOL DEL MAR II,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,6.2410,1.6176,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,CORDOSUC,NaN
1,2025-08-28,4RG6,AUTOG CELSIA SOLAR TQ JAMUNDI,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,INTERIOR,NaN
2,2025-08-28,4RGG,PARQUE SOLAR BARANOA,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,6.4200,2.2950,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,ATLANTIC,NaN
3,2025-08-28,4RMR,MINIGRANJA LA PAZ ESMERALDA,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.1523,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,GCM,NaN
4,2025-08-28,4S1G,MINIGRANJA VILLANUEVA,SOLAR,NO DESPACHADO CENTRALMENTE,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.4937,0.0228,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,GCM,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416,2025-08-28,2S9L,EL COCUYO,HIDRAULICA,NO DESPACHADO CENTRALMENTE,0.3543,0.3217,0.3047,0.2938,0.2776,...,0.1884,0.1881,0.1819,0.1943,0.2067,0.2078,0.2074,0.2060,INTERIOR,NaN
418,2025-08-28,2S8N,GUAVIO MENOR,HIDRAULICA,NO DESPACHADO CENTRALMENTE,5.0081,5.0249,5.0428,5.3112,5.3233,...,5.0748,4.9933,5.0089,5.0102,5.0156,5.0212,5.0207,4.9774,INTERIOR,NaN
419,2025-08-28,2S6S,AUTOG ARGOS YUMBO,TERMICA,NO DESPACHADO CENTRALMENTE,7.0380,8.2380,6.7650,5.5260,5.7090,...,4.7550,4.5150,4.4850,4.3050,4.4430,5.2650,5.5740,5.5530,INTERIOR,NaN
420,2025-08-28,2QRL,LA REBUSCA,HIDRAULICA,NO DESPACHADO CENTRALMENTE,0.6632,0.6635,0.6631,0.5850,0.6602,...,0.6592,0.6591,0.6590,0.6588,0.6588,0.6588,0.6590,0.6590,INTERIOR,NaN


In [61]:
# Obtener información de las plantas despachadas centralmente que tienen peso 
df_VariableDC=df_Variable[(df_Variable['Values_Name'].isin(['LATAM SOLAR LA LOMA','EL PASO','GUAYEPO','FUNDACION','CARACOLI I','PARQUE SOLAR LA UNION']))]
df_VariableDC=df_VariableDC.groupby(['RecCops','Values_Name','Values_Type','Values_Disp','subareafinal'])[l_per].mean().reset_index()
for col in l_per:
    df_VariableDC[col] = df_VariableDC[col].round(0)
    
df_Aux=df_VariableDC[['RecCops'] + l_per]
df_final=pd.concat([df_final,df_Aux],axis=0)

In [62]:
# Obtener información de las plantas despachadas centralmente que no tienen peso 
df_VariableOtros=df_Variable[(~df_Variable['Values_Name'].isin(['LATAM SOLAR LA LOMA','EL PASO','GUAYEPO','FUNDACION','CARACOLI I','PARQUE SOLAR LA UNION']))]
df_VariableOtros['Values_Type'] = df_VariableOtros['Values_Type'].apply(lambda x: x if x in ['EOLICA','SOLAR'] else 'NDC')
df_VariableOtros.loc[df_VariableOtros['Values_Disp'] == 'NO DESPACHADO CENTRALMENTE', 'Values_Type'] = 'NDC'
df_VariableOtros=df_VariableOtros.groupby(['Values_Name','Values_Type','Values_Disp','subareafinal'])[l_per].mean().reset_index()
df_VariableOtros=df_VariableOtros.groupby(['Values_Type','Values_Disp','subareafinal'])[l_per].sum().reset_index()

for col in l_per:
    df_VariableOtros[col] = df_VariableOtros[col].round(0)

df_VariableOtros['RecCops'] = df_VariableOtros['Values_Type'] + '_' + df_VariableOtros['subareafinal'].replace({
    'ATLANTIC': 'ATL',
    'BOLIVAR': 'BOL',
    'CERROMAT': 'CRR',
    'CORDOSUC': 'CS',
    'GCM': 'GCM',
    'INTERIOR': 'INT'
})

df_VariableOtros['RecCops']=df_VariableOtros['RecCops'].replace({'NDC_INT':'NDC'})

df_VariableOtros=df_VariableOtros[['RecCops'] + l_per]
df_final=pd.concat([df_final,df_VariableOtros],axis=0)

In [63]:
df_final.to_csv(os.path.join(r"C:\Alejo\cops\Modules\Utils\InformacionDia", f"DCOM_plt_{yearini}_{mesini}_{diaini}.csv"), index=False)